# 02 · Microeconomía del proyecto
**Pregunta:** ¿cómo interactúan escasez de stock, ritmo de absorción, caídas, mix y precio para un proyecto?

Este notebook separa **descripción**, **sensibilidad predictiva** y **causalidad**. Sin historial de precios por mes, no se declara elasticidad causal de precio.

In [ ]:
from pathlib import Path
import sys, pandas as pd, numpy as np, matplotlib.pyplot as plt
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src'/'replica_cygnus').exists())
sys.path.insert(0, str(ROOT/'src'))
from replica_cygnus.economic_intelligence import install_feature_mart, load_monthly_panel
from replica_cygnus.economic_intelligence.economics import price_area_indifference_proxy
install_feature_mart(); panel = load_monthly_panel()
PROJECT = panel['proyecto'].dropna().iloc[0]
d = panel.loc[panel['proyecto'].eq(PROJECT)].sort_values('periodo_mes').copy()

## Oferta, demanda y escasez
- **Oferta operativa:** stock disponible observado.
- **Demanda revelada:** separaciones netas de caídas.
- **Ventas/minutas:** conversión comercial posterior; no vuelve a consumir stock disponible.
- **Escasez:** meses de stock al ritmo reciente.

In [ ]:
d['meses_stock_ma3'] = d['saldo_final_observado'] / d['mov_neto_ma3'].replace(0, np.nan)
d[['periodo_mes','stock_inicio_observado','movimiento_neto_mes','ventas_minutas_mes','caidas_mes','absorcion_neta_mes','meses_stock_ma3']].tail(24)

In [ ]:
fig, ax1 = plt.subplots(figsize=(12,5))
ax1.bar(d['periodo_mes'], d['movimiento_neto_mes'], width=20, alpha=.45, label='Demanda neta')
ax1.plot(d['periodo_mes'], d['mov_neto_ma3'], label='Demanda neta MA3')
ax2 = ax1.twinx(); ax2.plot(d['periodo_mes'], d['saldo_final_observado'], linestyle='--', label='Stock fin')
ax1.set_title(f'{PROJECT} · Oferta vs demanda revelada')
ax1.legend(loc='upper left'); ax2.legend(loc='upper right'); plt.show()

## Precio y elección: lo que sí y lo que todavía no
`precio_m2_prom_actual_ref` y `descuento_prom_actual_ref` son referencias actuales. Sirven para comparar proyectos hoy, **no** para afirmar que un cambio histórico de precio causó un cambio de absorción.

Para elasticidad causal necesitamos snapshots mensuales de lista/precio final o un experimento comercial con variación exógena.

In [ ]:
latest = d.iloc[-1]
pd.Series({
 'precio_lista_prom_actual_ref': latest.get('precio_lista_prom_actual_ref'),
 'precio_m2_prom_actual_ref': latest.get('precio_m2_prom_actual_ref'),
 'descuento_prom_actual_ref': latest.get('descuento_prom_actual_ref'),
 'stock_total_actual_ref': latest.get('stock_total_departamentos_actual_ref'),
 'cobertura_ledger': latest.get('cobertura_oferta_ledger_vs_universo_actual')
})

## Curvas de indiferencia: proxy operativo
Sin elecciones individuales no estimamos utilidad estructural. Como herramienta comercial usamos **curvas iso-presupuesto**: combinaciones área × precio/m² que mantienen constante el precio total.

In [ ]:
iso = price_area_indifference_proxy(np.arange(45,121,5), budgets=[300000,400000,500000,600000])
fig, ax = plt.subplots(figsize=(9,5))
for budget, g in iso.groupby('budget'):
    ax.plot(g['area_m2'], g['price_m2_iso_budget'], label=f'S/ {budget:,.0f}')
ax.set_xlabel('Área m²'); ax.set_ylabel('S/ por m²'); ax.set_title('Curvas iso-presupuesto (proxy, no utilidad causal)'); ax.legend(); ax.grid(alpha=.2); plt.show()

### Decisiones que habilita
1. detectar proyectos con stock lento; 2. separar problema de demanda vs conversión; 3. priorizar tipologías para pricing; 4. diseñar experimentos de descuento; 5. definir guardrails de margen antes de recomendar precio.